In [5]:
import os
os.chdir("..")

import duckdb
con = duckdb.connect("data/clickstream.duckdb")

In [6]:
con.sql("""
SELECT
    COUNT(*) AS raw_rows,
    COUNT(*) FILTER (WHERE price IS NULL) AS null_price_rows,
    COUNT(*) FILTER (WHERE price <= 0) AS nonpositive_price_rows,
    COUNT(*) FILTER (WHERE price > 0) AS rows_after_price_filter,
    (SELECT COUNT(*) FROM stg_events) AS rows_in_stg_events,
    COUNT(*) FILTER (WHERE price > 0)
        - (SELECT COUNT(*) FROM stg_events) AS duplicates_removed
FROM read_parquet('data/parquet/events_20pct.parquet');
""").df()

,raw_rows,null_price_rows,nonpositive_price_rows,rows_after_price_filter,rows_in_stg_events,duplicates_removed
0,21923857,0,50675,21873182,21846166,27016


In [2]:
import os
os.chdir("..")

import duckdb
con = duckdb.connect("data/clickstream.duckdb")

In [3]:
con.sql("""
SHOW TABLES
""").df()

,name
0,events_sessionized
1,sessions
2,stg_events


In [4]:
con.sql("""
SELECT
    (SELECT COUNT(*) FROM stg_events) AS stg_rows,
    (SELECT COUNT(*) FROM events_sessionized) AS sessionized_rows,
    (SELECT SUM(n_events) FROM sessions) AS rows_in_sessions
""").df()

,stg_rows,sessionized_rows,rows_in_sessions
0,21846166,21846166,21846166.0


In [5]:
con.sql("""
SELECT
    COUNT(*) AS n_sessions,
    ROUND(AVG(n_events), 2) AS avg_events,
    MEDIAN(n_events) AS median_events,
    ROUND(100.0 * AVG((n_events = 1)::INT), 1) AS bounce_pct,
    ROUND(100.0 * AVG((n_purchases > 0)::INT), 2) AS conversion_pct,
    MAX(n_events) AS biggest_session
FROM sessions
""").df()

,n_sessions,avg_events,median_events,bounce_pct,conversion_pct,biggest_session
0,3741708,5.84,3.0,29.5,6.93,1707


In [6]:
con.sql("""
SELECT
    session_id,
    event_time,
    event_type,
    product_id
FROM events_sessionized
WHERE user_id = (
    SELECT user_id
    FROM sessions
    GROUP BY 1
    HAVING COUNT(*) BETWEEN 3 AND 6
    LIMIT 1
)
ORDER BY event_time
""").df()

,session_id,event_time,event_type,product_id
0,576629058_1,2019-11-25 16:57:48,view,12703494
1,576629058_1,2019-11-25 16:58:03,view,12703494
2,576629058_1,2019-11-25 16:58:37,view,12707153
3,576629058_1,2019-11-25 16:58:49,view,12703498
4,576629058_1,2019-11-25 16:59:06,view,12703493
5,576629058_1,2019-11-25 16:59:48,view,12709709
6,576629058_1,2019-11-25 17:00:15,view,12706857
7,576629058_1,2019-11-25 17:00:47,view,12706857
8,576629058_1,2019-11-25 17:01:01,view,12706857
9,576629058_1,2019-11-25 17:02:42,view,12709464


In [7]:
con.sql("""
SELECT
    (SELECT COUNT(*) FROM sessions) AS my_sessions,
    (SELECT COUNT(DISTINCT user_session) FROM stg_events) AS dataset_sessions
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,my_sessions,dataset_sessions
0,3741708,4578740


In [17]:
con.sql("""
WITH x AS (
    SELECT user_session, COUNT(DISTINCT session_id) AS my_sessions_inside
    FROM events_sessionized GROUP BY 1
)
SELECT my_sessions_inside, COUNT(*) AS dataset_sessions
FROM x GROUP BY 1 ORDER BY 1 LIMIT 8
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,my_sessions_inside,dataset_sessions
0,1,4497665
1,2,70605
2,3,7214
3,4,1719
4,5,651
5,6,340
6,7,182
7,8,99


In [18]:
con.sql("""
WITH x AS (
    SELECT session_id, COUNT(DISTINCT user_session) AS dataset_ids_inside
    FROM events_sessionized GROUP BY 1)
SELECT dataset_ids_inside, COUNT(*) AS my_sessions
FROM x GROUP BY 1 ORDER BY 1 LIMIT 8
""").df()

,dataset_ids_inside,my_sessions
0,0,1
1,1,3160115
2,2,416605
3,3,99229
4,4,33247
5,5,13680
6,6,6650
7,7,3777


In [19]:
res = con.sql("""
WITH gaps AS (
    SELECT event_time - LAG(event_time) OVER (
               PARTITION BY user_id ORDER BY event_time, product_id, event_type) AS gap
    FROM stg_events
)
SELECT
    COUNT(*) FILTER (WHERE gap IS NULL OR gap > INTERVAL 15 MINUTE) AS sessions_15min,
    COUNT(*) FILTER (WHERE gap IS NULL OR gap > INTERVAL 30 MINUTE) AS sessions_30min,
    COUNT(*) FILTER (WHERE gap IS NULL OR gap > INTERVAL 60 MINUTE) AS sessions_60min
FROM gaps
""").df()
res

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,sessions_15min,sessions_30min,sessions_60min
0,3929368,3741708,3588440


In [20]:
base = res.loc[0, "sessions_30min"]

for c in res.columns:
    print(
        c,
        res.loc[0, c],
        f"{100 * (res.loc[0, c] / base - 1):+.1f}% vs 30 min"
    )

sessions_15min 3929368 +5.0% vs 30 min
sessions_30min 3741708 +0.0% vs 30 min
sessions_60min 3588440 -4.1% vs 30 min
